# 65 — Documentary Evidence Fusion (non-empty fallback)

This version always writes a non-empty site-year evidence table by using
configured sites from `run.yml` and years inferred from linked data or
analysis dates.

In [ ]:
import os, json, hashlib, platform, sys, re
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        if path.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(path), "status": "empty_file"}
        df = pd.read_csv(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest: dict, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

In [ ]:
step = "65_documentary_evidence_fusion"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

linked_emissions, meta_e = safe_read_csv(OUTPUTS / "64_documentary_site_year_linkage" / "linked_emissions.csv")
linked_cems, meta_c = safe_read_csv(OUTPUTS / "64_documentary_site_year_linkage" / "linked_cems.csv")
linked_catalog, meta_k = safe_read_csv(OUTPUTS / "64_documentary_site_year_linkage" / "linked_catalog.csv")
run_cfg = load_yaml(CONFIGS / "run.yml")
sites = pd.DataFrame(run_cfg.get("sites", []))
if not sites.empty:
    sites["site_id"] = sites.get("id", pd.Series(dtype=str)).astype(str)
    sites["site_name"] = sites.get("name", pd.Series(dtype=str)).astype(str)
else:
    sites = pd.DataFrame(columns=["site_id","site_name"])

years = set()
for df in [linked_emissions, linked_cems, linked_catalog]:
    if not df.empty:
        for c in ["report_year", "year"]:
            if c in df.columns:
                years |= set(pd.to_numeric(df[c], errors="coerce").dropna().astype(int).tolist())

if not years:
    run_section = run_cfg.get("run", {})
    y1 = pd.to_datetime(run_section.get("date_from", "2025-01-01"), errors="coerce").year
    y2 = pd.to_datetime(run_section.get("date_to", "2025-12-31"), errors="coerce").year
    years = set(range(min(y1, y2), max(y1, y2) + 1))

site_ids = set()
for df in [linked_emissions, linked_cems, linked_catalog]:
    if not df.empty and "site_id" in df.columns:
        site_ids |= set(df["site_id"].dropna().astype(str).tolist())
if not site_ids and not sites.empty:
    site_ids = set(sites["site_id"].astype(str).tolist())

rows = []
for site_id in sorted(site_ids):
    site_name = site_id
    if not sites.empty:
        hit = sites[sites["site_id"].astype(str) == str(site_id)]
        if not hit.empty:
            site_name = hit["site_name"].iloc[0]

    for year in sorted(years):
        e = linked_emissions[(linked_emissions["site_id"].astype(str) == str(site_id)) & (pd.to_numeric(linked_emissions.get("report_year", np.nan), errors="coerce") == year)] if not linked_emissions.empty and "site_id" in linked_emissions.columns else pd.DataFrame()
        c = linked_cems[(linked_cems["site_id"].astype(str) == str(site_id)) & (pd.to_numeric(linked_cems.get("report_year", np.nan), errors="coerce") == year)] if not linked_cems.empty and "site_id" in linked_cems.columns else pd.DataFrame()
        k = linked_catalog[(linked_catalog["site_id"].astype(str) == str(site_id))] if not linked_catalog.empty and "site_id" in linked_catalog.columns else pd.DataFrame()

        emissions_rows = len(e)
        exceedance_rows = int(pd.to_numeric(e.get("limit_exceedance_flag", 0), errors="coerce").fillna(0).sum()) if emissions_rows else 0
        cems_rows = len(c)
        noncomp_rows = int(pd.to_numeric(c.get("noncompliance_events", 0), errors="coerce").fillna(0).sum()) if cems_rows else 0
        report_count = len(k)

        documentary_score = (
            min(emissions_rows, 20) * 0.1
            + min(exceedance_rows, 10) * 0.5
            + min(cems_rows, 20) * 0.1
            + min(noncomp_rows, 10) * 0.5
            + min(report_count, 10) * 0.1
        )

        rows.append({
            "site_id": str(site_id),
            "site_name": str(site_name),
            "report_year": int(year),
            "documentary_emissions_rows": int(emissions_rows),
            "documentary_exceedance_rows": int(exceedance_rows),
            "documentary_cems_rows": int(cems_rows),
            "documentary_noncompliance_rows": int(noncomp_rows),
            "documentary_report_count": int(report_count),
            "documentary_evidence_score": round(float(documentary_score), 3),
        })

fused = pd.DataFrame(rows)
if fused.empty:
    raise RuntimeError("Could not construct documentary evidence rows even from configured sites and analysis years")

fused.to_csv(out / "documentary_evidence_site_year.csv", index=False)
write_json(out / "documentary_evidence_summary.json", {
    "rows": int(len(fused)),
    "sites": int(fused["site_id"].nunique()),
    "years": int(fused["report_year"].nunique()),
    "nonzero_documentary_score_rows": int((pd.to_numeric(fused["documentary_evidence_score"], errors="coerce").fillna(0) > 0).sum()),
})

manifest = manifest_base(step, [CONFIGS / "documentary_sources.yml", CONFIGS / "run.yml"])
manifest["inputs"] = {"linked_emissions": meta_e, "linked_cems": meta_c, "linked_catalog": meta_k}
add_artifact(manifest, out / "documentary_evidence_site_year.csv")
add_artifact(manifest, out / "documentary_evidence_summary.json")
write_json(out / "manifest.json", manifest)

display(fused.head(20))